# Evaluating conformal predictions

Evaluation is independent of the estimator. The evaluators receive observed targets and predictions that have already been produced.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from tinyconformal.evaluation import PanelEvaluator, RegressorEvaluator

## Tabular regression

`RegressorEvaluator` consumes the two-column array returned by `predict_interval()`.

In [2]:
y_test = np.array([10.0, 12.0, 15.0, 18.0])
intervals = np.array([
    [8.0, 12.0],
    [10.0, 14.0],
    [13.0, 17.0],
    [14.0, 17.0],
])

RegressorEvaluator.evaluate(
    y_true=y_test,
    intervals=intervals,
    coverage=0.9,
)

,coverage,coverage_rate,interval_width_mean,mwis,n_obs
0,0.9,0.75,3.75,8.75,4


## Panel forecasts

`PanelEvaluator` aligns observations and forecasts using `unique_id` and `ds`. Interval pairs following the `<model>-lo-<coverage>` and `<model>-hi-<coverage>` convention are discovered automatically.

In [3]:
forecast = pd.DataFrame({
    "unique_id": ["a", "a", "b", "b"],
    "ds": pd.to_datetime(["2026-01-01", "2026-01-02"] * 2),
    "LinearRegression": [10.0, 12.0, 15.0, 16.0],
    "LinearRegression-lo-90": [8.0, 10.0, 13.0, 14.0],
    "LinearRegression-hi-90": [12.0, 14.0, 17.0, 18.0],
})

observed = pd.DataFrame({
    # Deliberately use a different order to demonstrate key-based alignment.
    "unique_id": ["b", "a", "b", "a"],
    "ds": pd.to_datetime(["2026-01-02", "2026-01-01", "2026-01-01", "2026-01-02"]),
    "y": [19.0, 10.0, 15.0, 12.0],
})

PanelEvaluator.evaluate(
    y_true=observed,
    forecast=forecast,
)

,model,coverage,coverage_rate,interval_width_mean,mwis,n_obs
0,LinearRegression,0.9,0.75,4.0,9.0,4.0


## Custom interval-column names

When columns do not follow the standard convention, declare each pair explicitly:

In [4]:
custom_forecast = forecast.rename(columns={
    "LinearRegression-lo-90": "lower",
    "LinearRegression-hi-90": "upper",
})

PanelEvaluator.evaluate(
    y_true=observed,
    forecast=custom_forecast,
    intervals={"LinearRegression": ("lower", "upper", 0.9)},
)

,model,coverage,coverage_rate,interval_width_mean,mwis,n_obs
0,LinearRegression,0.9,0.75,4.0,9.0,4.0
